<div style="padding:20px;color:white;margin:0;font-size:300%;text-align:center;display:fill;border-radius:60px;background-color:#680EAB;overflow:hidden;font-weight:800">Wingo Color Prediction</div>

_______________

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>1  |  About Dataset</div></b>

**This is a Win and Go type contest where one have to predict the color which upon predicting correctly will double or tripple the money( Depending on the money you bet for). There is some data generated along with the color where we can make use of it to predict the color.**

## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>2  |  Import Libraries</div></b>

In [ ]:
# importing libraries for data processing and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


# importing libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder


# ignore warnings
import warnings
warnings.filterwarnings('ignore')

# full display of columns and rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>3  |  Download Dataset</div></b>

In [ ]:
df = pd.read_csv("/kaggle/input/wingo-color-prediction/Win-Go.csv.csv")

In [ ]:
df.head()

## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>4  |  Inspecting and Cleaning Dataframe Structure</div></b>

In [ ]:
# Dataframe summary

def summary(df):
    print(f'data shape: {df.shape}')
    summ = pd.DataFrame(df.dtypes, columns=['Data Type'])
    summ['Missing#'] = df.isna().sum()
    summ['Missing%'] = (df.isna().sum())/len(df)
    summ['Dups'] = df.duplicated().sum()
    summ['Uniques'] = df.nunique().values
    summ['Count'] = df.count().values
    desc = pd.DataFrame(df.describe(include='all').transpose())
    summ['Min'] = desc['min'].values
    summ['Max'] = desc['max'].values
    summ['Average'] = desc['mean'].values
    summ['Standard Deviation'] = desc['std'].values
    summ['First Value'] = df.loc[0].values
    summ['Second Value'] = df.loc[1].values
    summ['Third Value'] = df.loc[2].values

    display(summ)

summary(df)

<div class="alert alert-block alert-info" style="background-color:#DCECFD;color:#680EAB;border-color:black;width:80%;margin: auto;text-align: center;"><b>Comment:</b> No NaN, no Dups</div>

## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>5  |  EDA</div></b>

In [ ]:
# Features histogram
for col in df.columns:
    plt = px.histogram(df, x = col, title=col, color_discrete_sequence=px.colors.sequential.Cividis)
    plt.show()

In [ ]:
# Features Histogram vs Target
for col in df.columns:
    plt = px.histogram(df, x = col, color ="Color", title=col + ' vs Color', color_discrete_sequence=px.colors.qualitative.Light24)
    plt.show()

In [ ]:
sns.scatterplot(data=df, x="Number", y="Color")

In [ ]:
pd.crosstab(df["Color"], df["Number"])

**As we can see from the table above, each number represents a color, as shown below:**
* Green = 1, 3, 7, 9
* Green-Violet = 5
* Red = 2, 4, 6, 8
* Red-Violet = 0

**Therefore, it is not necessary to use algorithms to predict colors, just use the table above.
But just for educational purposes we will use the XGBClassifier to confirm, where Accuracy will be 1.00 (100%).**


## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>6  |  XGBClassifier</div></b>

In [ ]:
# Convert the columns to numeric.
le = LabelEncoder()
for column in df.columns:
    if df[column].dtype == type(object):
        df[column] = le.fit_transform(df[column])

In [ ]:
# Heatmap and Correlation Matrix
import matplotlib.pyplot as plt

corr = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
df.info()

In [ ]:
# Create a copy of df
df_copy = df.copy(deep=True)

In [ ]:
# Split data into Features and Target
X = df_copy.drop(['Color'], axis=1)
y = df_copy['Color']

In [ ]:
# Split Data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Scaling the Data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Fit
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from xgboost import XGBClassifier

xgbc=XGBClassifier()
xgbc.fit(X_train,y_train)

In [ ]:
# Predict the Test set results

y_pred = xgbc.predict(X_test)

`Accuracy Score` Accuracy is the percentage of data that are correctly classified, which ranges from 0 to 1. This measure is quite instinctive that we just compare the predicted class and the actual class, and we want the model to correctly classify the data.

In [ ]:
# Check accuracy score 

from sklearn.metrics import accuracy_score

print('Model accuracy score : {0:0.4f}'. format(accuracy_score(y_test, y_pred)))

In [ ]:
# ConfusionMatrix

from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap = plt.cm.Purples, normalize = None, display_labels = ['Green', 'Green-Violet', 'Red', 'Red_Violet'])

In [ ]:
# classification_report

print(classification_report(y_test, y_pred))

In [ ]:
# classification_report_imbalanced

from imblearn.metrics import classification_report_imbalanced

print(classification_report_imbalanced(y_test,y_pred))

## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>7  |  RandomForestRegressor</div></b>

**Now let's use RandomForestRegressor to predict the Price.
As we saw in the Correlation Matrix, the Price has an almost perfect correlation with the Period. Let's confirm this.**

In [ ]:
df.head()

In [ ]:
# The first six digits of Period are always the same, so let's delete them.
df['Period'] = df['Period'].astype(str).str[6:]

df.head()

In [ ]:
X = df.drop(columns=['Price'])
y = df['Price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
import sklearn

random_forest = RandomForestRegressor()
random_forest.fit(X_train, y_train)
preds = random_forest.predict(X_train)
acc_random_forest = sklearn.metrics.r2_score(y_train,preds)
print('R2 Score: ', acc_random_forest)

**As expected, R2 is practically 1.**

In [ ]:
df_final = pd.DataFrame(columns=["Predicted","Price"])
df_final["Price"] = y_test
random_forest = RandomForestRegressor()
random_forest.fit(X_train, y_train)
df_final["Predicted"] = random_forest.predict(X_test).round(1)
df_final["Price"] = df_final["Price"]
df_final.head()

**We can see that the predictions are very close to the original Price value.**

## <b><div style='padding:15px;background-color:#680EAB;color:white;border-radius:40px;font-size:110%;text-align: center'>If you liked this code, consider upvoting it. Thank you.</div></b>